# Gemma: public source, private inputs, full Kaggle GPU rerun

Attach **native inputs và `gemma_private_runtime.json` dưới dạng private Kaggle inputs (xem README_VI.md)**, bật Internet và GPU T4. Code đã nhúng trong notebook; không cần repo cha hoặc cache kết quả cũ. ZIP và cây thư mục do Kaggle tự giải nén đều được nhận và kiểm manifest.

**Trên Kaggle: attach data, chọn GPU T4, bật Internet rồi Run All.** Hai biến install/run mặc định bật khi ở Kaggle; laptop không chạy phần nặng. Full route: encode5817 ảnh/20065 texts, validation grid, refit ba seeds, test2000-replicate CI, bốn ablations, timing128 ảnh → CSV/PNG/SVG → báo cáo tiếng Việt HTML/Markdown → ZIP tải về. Không tự publish notebook hoặc thay kết quả đã lưu.

Báo cáo phân biệt kết quả mới với ngữ cảnh bảy backbone ngày16/10/2026. Không huấn luyện lại bảy backbone. Model weights tải đúng revision/SHA từ Hugging Face; nếu model yêu cầu quyền truy cập, chấp nhận điều khoản và thêm Kaggle Secret `HF_TOKEN`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
ON_KAGGLE = Path('/kaggle/working').is_dir()
INSTALL_GPU_DEPENDENCIES = ON_KAGGLE
RUN_FULL_GPU = ON_KAGGLE
RUN_TIMING_128 = True
NATIVE_DATA = None  # Auto-discover native_inputs.zip / expanded snapshot under /kaggle/input.
RUN_NAME = 'gemma_e2e_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')

In [ ]:
import os, sys, io, base64, tempfile, zipfile
from pathlib import Path
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
roots=[Path.cwd(),Path.cwd().parent,Path.cwd()/'gemma_research_standalone']
CODE_ROOT=next((p.resolve() for p in roots if (p/'src/bundle.py').is_file() and (p/'PINNED.json').is_file()),None)
if CODE_ROOT is None:
    CODE_ROOT=Path(tempfile.mkdtemp(prefix='gemma_standalone_code_'))
    with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
        for member in archive.infolist():
            if not (CODE_ROOT/member.filename).resolve().is_relative_to(CODE_ROOT.resolve()):
                raise ValueError('Embedded source path escaped destination')
        archive.extractall(CODE_ROOT)
sys.path.insert(0,str(CODE_ROOT/'src'))
from bundle import verify_code
pins=verify_code()
if globals().get('RUN_FULL_GPU',False):
    from bundle import require_private_config
    require_private_config()
print('Frozen scientific source verified. Model:',pins['gemma_model']['repo'])

In [ ]:
if RUN_FULL_GPU:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Kaggle Settings -> Accelerator -> GPU T4, then restart session and Run All.')
    print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,'CUDA:',torch.version.cuda)
    print('One GPU is used. Recommended: T4 16GB; the laptop is not used for computation.')
    if ON_KAGGLE and not os.environ.get('HF_TOKEN'):
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
            if token: os.environ['HF_TOKEN']=token
            del token
        except Exception:
            pass  # Public access may work; never print secrets or exception contents.
if INSTALL_GPU_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable,'-m','pip','install','--no-input','--retries','2','--timeout','30','-r',str(CODE_ROOT/'requirements_gpu.txt')],check=True,timeout=600)
else:
    print('Dependency installation disabled. Enable it before a full GPU rerun if the pinned stack is not installed.')

## Encode → validation-only selection → refit → test → CI/ablation → timing → report

Native Gemma FP32 cố định; cache/head FP32. Source `learning.py`/`run.py` giữ nguyên byte. Wrapper chỉ inject explicit FP32 Encoder và quản lý input/output. Cùng protocol không bảo đảm bitwise-identical result ở GPU/runtime khác; mọi model weight SHA, recipe và runtime được ghi receipt.

In [ ]:
if RUN_FULL_GPU:
    import importlib.metadata as metadata
    required={'numpy':'2.3.5','scipy':'1.17.0','transformers':'5.19.0','sentence-transformers':'6.1.0','huggingface_hub':'1.33.0'}
    for package,version in required.items():
        assert metadata.version(package)==version, f'Enable INSTALL_GPU_DEPENDENCIES for {package}=={version}'
    import subprocess
    OUTPUT=(Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'runs')/RUN_NAME
    command=[sys.executable,'-u',str(CODE_ROOT/'src/run_native.py'),'--output',str(OUTPUT)]
    if NATIVE_DATA:command+=['--data',str(NATIVE_DATA)]
    if not RUN_TIMING_128:command+=['--skip-timing']
    # Fresh interpreter uses installed package versions even if Kaggle preloaded NumPy.
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line,end='',flush=True)
    if process.wait()!=0:raise RuntimeError('GPU pipeline failed; see streamed logs and execution_status.json. Completed evidence is retained.')
    print('Full Gemma end-to-end run complete:',OUTPUT)
else:
    print('Full native run disabled. Set INSTALL_GPU_DEPENDENCIES=True and RUN_FULL_GPU=True, attach inputs, then Run All on a GPU.')

In [ ]:
if RUN_FULL_GPU:
    from bundle import read
    from common import verify
    verify(OUTPUT)
    assert read(OUTPUT/'execution_status.json')['complete']
    assert read(OUTPUT/'core/status.json')['complete']
    m=read(OUTPUT/'core/gemma/metrics.json')
    for name in ['base','mean_single','uncertainty_single']:print(name,m[name])
    if RUN_TIMING_128:
        timing=OUTPUT/'timing128'
        verify(timing)
        assert read(timing/'bootstrap_status.json')['complete']
        assert len(read(timing/'timing_results.json'))==6
        assert len(read(timing/'timing_samples.json'))==864
        print('128-image native timing verified: six rows / 864 calls.')
    verify(OUTPUT/'report')
    audit=read(OUTPUT/'report/report_audit.json')
    assert audit['complete'] and audit['evidence_label']=='new_native_gpu_run'
    print('Audited report:',OUTPUT/'report/REPORT_VI.html')

## Báo cáo, biểu đồ và tải toàn bộ kết quả

Mỗi lượt chạy tạo một folder riêng: `core/`, `timing128/`, `report/`, code snapshot, execution receipt và manifest SHA256. ZIP cạnh folder chứa toàn bộ kết quả, gồm cache mới/checkpoints/predictions; ảnh input và pretrained weights không được copy vào ZIP. HTML tự chứa hình, có thể xem offline hoặc dùng Print → Save as PDF.

In [ ]:
if RUN_FULL_GPU:
    from IPython.display import display, Markdown, Image, FileLink
    report_lines=(OUTPUT/'report/REPORT_VI.md').read_text(encoding='utf-8').splitlines()
    display(Markdown(chr(10).join(line for line in report_lines if not line.startswith('!['))))
    for path in sorted((OUTPUT/'report/figures').glob('*.png')):
        display(Image(filename=str(path)))
    print('Download from Kaggle Output / Save Version:')
    for path in [OUTPUT/'report/REPORT_VI.html',OUTPUT/'report/REPORT_VI.md',OUTPUT.with_name(OUTPUT.name+'_results.zip')]:
        display(FileLink(str(path)))

## Giữ output và diễn giải

Save Version (Save & Run All) để Kaggle giữ output ngay cả khi laptop tắt: báo cáo, biểu đồ, CSV, ZIP, records/mapping, native cache, checkpoints, metrics, paired CI, validation trials, training histories và raw timing. Mean seed11 vẫn primary; uncertainty/ensemble/ablation không được dùng để chọn lại theo test.

Có thể rebuild public inputs bằng `vendor/stage11/prepare_data.py` khi cần protocol mới/khôi phục; snapshot đính kèm là đường chạy gốc đã kiểm SHA. Số liệu hiện tại exploratory, không fresh holdout/SOTA hoặc exact Stage9/10 recovery.